# Inferencia de YOLO11 en CPU

Este notebook carga el modelo vehicular entrenado, fuerza el uso exclusivo de CPU, ejecuta predicciones sobre imágenes de ejemplo y mide la latencia. No entrena ni modifica el modelo.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Debe ejecutarse antes de importar torch

import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

DEVICE = "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Dispositivo obligatorio: {DEVICE}")
print(f"CUDA visible para este proceso: {torch.cuda.is_available()}" " - (no utilizada; inferencia forzada en CPU)")


## Rutas relativas y validación de archivos

El notebook funciona tanto si Jupyter se inició desde la raíz del repositorio como desde `notebooks/`. No contiene rutas absolutas de una computadora específica.

In [ ]:
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

MODEL_PATH = ROOT / "runs" / "entrenamiento" / "yolo11n_vehiculos_definitivo_seed42" / "weights" / "best.pt"
TEST_IMAGES = ROOT / "vehicle dataset limpio final" / "images" / "test"

assert MODEL_PATH.is_file(), f"No se encontró el modelo: {MODEL_PATH}"
assert TEST_IMAGES.is_dir(), f"No se encontró el directorio de imágenes: {TEST_IMAGES}"

extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
imagenes_test = sorted(p for p in TEST_IMAGES.iterdir() if p.suffix.lower() in extensiones)
assert imagenes_test, "No se encontraron imágenes de prueba"
print(f"Modelo: {MODEL_PATH.relative_to(ROOT)}")
print(f"Imágenes disponibles: {len(imagenes_test)}")

## Carga explícita con `map_location="cpu"`

Primero se verifica que el checkpoint puede deserializarse directamente en CPU. Después Ultralytics construye el detector y se comprueba el dispositivo real de sus parámetros. `weights_only=False` es apropiado aquí porque `best.pt` es un archivo local y confiable producido por nuestro propio entrenamiento.

In [ ]:
checkpoint_cpu = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
print(f"Checkpoint cargado en CPU: {type(checkpoint_cpu).__name__}")
del checkpoint_cpu

modelo = YOLO(str(MODEL_PATH))
modelo.to(DEVICE)
dispositivo_real = next(modelo.model.parameters()).device.type
assert dispositivo_real == "cpu", f"El modelo quedó en {dispositivo_real}"
print(f"Dispositivo real de los parámetros: {dispositivo_real}")
print(f"Clases: {modelo.names}")

## Selección reproducible de seis ejemplos

Se toman posiciones equidistantes de la lista ordenada, evitando una elección manual favorable.

In [ ]:
indices = np.linspace(0, len(imagenes_test) - 1, num=min(6, len(imagenes_test)), dtype=int)
muestra = [imagenes_test[i] for i in indices]
pd.DataFrame({"indice": indices, "imagen": [p.name for p in muestra]})

## Calentamiento y benchmark en CPU

El calentamiento evita que la primera inicialización distorsione la medición. La latencia reportada incluye la llamada completa de predicción para cada imagen.

In [ ]:
for _ in range(3):
    _ = modelo.predict(source=str(muestra[0]), device=DEVICE, imgsz=640, conf=0.25, verbose=False)

resultados = []
latencias_ms = []
for ruta in muestra:
    inicio = time.perf_counter()
    resultado = modelo.predict(source=str(ruta), device=DEVICE, imgsz=640, conf=0.25, verbose=False)[0]
    latencias_ms.append((time.perf_counter() - inicio) * 1000)
    resultados.append(resultado)

resumen_cpu = pd.DataFrame({
    "metrica": ["latencia media (ms)", "mediana (ms)", "p95 (ms)", "FPS equivalente"],
    "valor": [np.mean(latencias_ms), np.median(latencias_ms), np.percentile(latencias_ms, 95), 1000 / np.mean(latencias_ms)],
})
resumen_cpu.round(3)

## Predicciones obtenidas

In [ ]:
filas = []
for ruta, resultado, latencia in zip(muestra, resultados, latencias_ms):
    if resultado.boxes is None or len(resultado.boxes) == 0:
        filas.append({"imagen": ruta.name, "clase": "sin detecciones", "confianza": np.nan, "latencia_ms": latencia})
        continue
    for clase, confianza in zip(resultado.boxes.cls.cpu().tolist(), resultado.boxes.conf.cpu().tolist()):
        filas.append({"imagen": ruta.name, "clase": modelo.names[int(clase)], "confianza": confianza, "latencia_ms": latencia})

df_predicciones = pd.DataFrame(filas)
df_predicciones

## Visualización

Las cajas, clases y confianzas son producidas por el modelo ejecutado exclusivamente en CPU.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, ruta, resultado, latencia in zip(axes.ravel(), muestra, resultados, latencias_ms):
    imagen_bgr = resultado.plot()
    ax.imshow(imagen_bgr[..., ::-1])
    ax.set_title(f"{ruta.name}\nCPU: {latencia:.1f} ms")
    ax.axis("off")
for ax in axes.ravel()[len(resultados):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Conclusión

El checkpoint se cargó explícitamente con `map_location="cpu"`, todos los parámetros permanecieron en CPU y las seis imágenes se procesaron sin depender de CUDA. La latencia medida permite estimar la viabilidad del modelo en equipos sin GPU. Para una comparación justa entre computadoras deben mantenerse `imgsz=640`, `conf=0.25`, el mismo checkpoint y el mismo conjunto de imágenes.